# 🚩 Notebook 2: Feature Flags

A **feature flag** lets you toggle behaviour at runtime — without a deploy.
Uses:
- Hide an unfinished feature behind `off`.
- Canary to 5% of users.
- Emergency kill-switch for a buggy feature.

## 🛠️ Setup

```bash
cd 05-microservices/configuration-externalization
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
import hashlib, json

class FlagStore:
    """Pretends to be a remote config service. In prod this would be LaunchDarkly, Unleash, etc."""
    def __init__(self, flags): self.flags = flags
    def update(self, new): self.flags.update(new)  # hot reload
    def enabled(self, flag, user_id=None):
        f = self.flags.get(flag)
        if f is None: return False
        if isinstance(f, bool): return f
        if isinstance(f, dict) and 'percent' in f and user_id is not None:
            h = int(hashlib.md5(f'{flag}:{user_id}'.encode()).hexdigest(), 16) % 100
            return h < f['percent']
        return False

flags = FlagStore({
    'dark_mode': True,           # everyone
    'new_checkout': {'percent': 10},  # 10% canary
    'experimental_search': False,
})

for uid in range(5):
    print(f'user {uid}: dark_mode={flags.enabled("dark_mode",uid)} new_checkout={flags.enabled("new_checkout",uid)}')


### Kill-switch without a redeploy

In [ ]:
# Ops sees a bug in new_checkout. Turn it off *instantly*:
flags.update({'new_checkout': False})
print('after kill-switch:')
for uid in range(5):
    print(f'user {uid}: new_checkout={flags.enabled("new_checkout",uid)}')


### Environment-based config vs flags
| | Environment var | Feature flag |
|--|--|--|
| Change requires deploy? | yes (restart) | no (hot reload) |
| Per-user targeting? | hard | easy |
| Good for secrets? | yes | no |
| Good for experiments? | no | yes |